In [1]:
def generate_random_math_question():
    import random

    operations = ['+', '-', '*', '/']
    operation = random.choice(operations)

    if operation == '+':
        num1 = random.randint(1, 10)
        num2 = random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 + num2
        }
    elif operation == '-':
        num1 = random.randint(2, 10)
        num2 = random.randint(1, num1) # For multiplication, we keep it simple so num1 >= num2
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 - num2
        }
    elif operation == '*':
        num1 = random.randint(1, 10)
        num2 = random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": num1 * num2
        }
    elif operation == '/':
        # Ensure no division by zero and integer result
        num2 = random.randint(1, 10)
        num1 = num2 * random.randint(1, 10)
        output = {
            "operation": operation,
            "a": num1,
            "b": num2,
            "answer": int(num1 / num2)
        }

    return output

In [2]:
generate_random_math_question()

{'operation': '+', 'a': 1, 'b': 2, 'answer': 3}

In [24]:
import os
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

# load in the .env variables
load_dotenv()

def rag_document_ingestion(textfile):
    """Chunk documents from text file for RAG ingestion."""
    with open(textfile) as f:
        document = f.read()
    
    # Initialize Text Splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len
    )

    # Create Documents (Chunks) From File
    texts = text_splitter.create_documents([document])
    return texts


def vector_store_name_detection(topic):
    """Generate Vector Store name based on topic."""
    vector_store_name = topic.lower().replace(" ", "_") + "_knowledge_base"
    return vector_store_name

def vector_store_creation_based_on_extrapolated_topic(texts):
    """Create a Vector Store and persist it based on the determined topic of the read text"""


def vector_store_ingestion(texts, vector_store_name):
    """Ingest documents into Vector Store."""
    # Initialize OpenAI Embeddings
    embeddings = OpenAIEmbeddings(
        model=str(os.getenv("TEXT_EMBEDDING_MODEL")),
        api_key=str(os.getenv("API_KEY"))
    )

    # Initialize ChromaDB as Vector Store
    vector_store = Chroma(
        collection_name=vector_store_name,
        embedding_function=embeddings,
        persist_directory=f"./app/data/seed_context/{vector_store_name}"
    )

    # Add Documents to Vector Store
    vector_store.add_documents(texts)
    return vector_store

In [27]:
extracted_documents = rag_document_ingestion(r"app\data\seed_context\football.txt")

In [28]:
base_name = vector_store_name_detection("Football")

In [29]:
vector_store = vector_store_ingestion(extracted_documents, base_name)

In [23]:
# Query the Vector Store
results = vector_store.similarity_search(
    "What do players do on the field in football?",
    k=2
)

# Print Resulting Chunks
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n\n")

* Each team has players working together on the field.
Football matches are played in two halves. [{}]


* Football is a team sport played with a round ball.
Players try to score goals by kicking the ball into a goal.
Each team has players working together on the field.
Football matches are played in two halves.
A goal is scored when the ball goes into the net.
Players pass the ball to teammates.
Some players are defenders, midfielders, or strikers.
Football teams often wear matching jerseys.
Fans cheer when their team scores a goal.
Football is played all around the world. [{}]


